# CASPER Training Monitor

Run this notebook to visualize training progress in real-time.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import json

# Paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / 'checkpoints'
PROGRESS_FILE = CHECKPOINT_DIR / 'training_progress.csv'
CONV_LOG = CHECKPOINT_DIR / 'training_conversations.jsonl'

print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Progress file exists: {PROGRESS_FILE.exists()}")

In [ ]:
# Load training progress
df = pd.read_csv(PROGRESS_FILE)
print(f"Loaded {len(df)} episodes")
print(f"\nLatest stats (last 10 episodes):")
print(f"  Reward: {df['reward'].tail(10).mean():.4f}")
print(f"  NDCG:   {df['ndcg'].tail(10).mean():.4f}")
print(f"  Loss:   {df['loss'].tail(10).mean():.4f}")
df.tail(10)

In [ ]:
# Plot training curves
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Reward (per episode)', 'NDCG@10', 'Loss', 'Moving Averages'),
    vertical_spacing=0.12
)

# Reward
fig.add_trace(
    go.Scatter(x=df['episode'], y=df['reward'], mode='lines', name='Reward', 
               opacity=0.4, line=dict(color='blue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df['episode'], y=df['reward_ma10'], mode='lines', name='Reward MA10',
               line=dict(color='blue', width=2)),
    row=1, col=1
)

# NDCG
fig.add_trace(
    go.Scatter(x=df['episode'], y=df['ndcg'], mode='lines', name='NDCG',
               opacity=0.4, line=dict(color='green')),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=df['episode'], y=df['ndcg_ma10'], mode='lines', name='NDCG MA10',
               line=dict(color='green', width=2)),
    row=1, col=2
)

# Loss
fig.add_trace(
    go.Scatter(x=df['episode'], y=df['loss'], mode='lines', name='Loss',
               line=dict(color='red')),
    row=2, col=1
)

# Combined MAs
fig.add_trace(
    go.Scatter(x=df['episode'], y=df['reward_ma10'], mode='lines', name='Reward MA10',
               line=dict(color='blue', width=2)),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=df['episode'], y=df['ndcg_ma10'], mode='lines', name='NDCG MA10',
               line=dict(color='green', width=2)),
    row=2, col=2
)

fig.update_layout(height=700, showlegend=True, title_text='CASPER Training Progress')
fig.update_xaxes(title_text='Episode')
fig.show()

In [ ]:
# Load conversation logs for detailed analysis
conversations = []
if CONV_LOG.exists():
    with open(CONV_LOG, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                conversations.append(json.loads(line))
            except:
                pass

print(f"Loaded {len(conversations)} conversation logs")

# Analyze preferences
if conversations:
    liked_counts = []
    disliked_counts = []
    not_seen_counts = []
    eval_targets = []
    
    for conv in conversations:
        prefs = conv.get('preferences', {})
        liked_counts.append(len([x for x in prefs.get('liked', []) if x != 'unknown']))
        disliked_counts.append(len([x for x in prefs.get('disliked', []) if x != 'unknown']))
        not_seen_counts.append(len([x for x in prefs.get('not_seen', []) if x != 'unknown']))
        eval_targets.append(conv.get('num_eval_targets', 0))
    
    print(f"\nPreference extraction stats (last 20):")
    print(f"  Avg liked: {np.mean(liked_counts[-20:]):.1f}")
    print(f"  Avg disliked: {np.mean(disliked_counts[-20:]):.1f}")
    print(f"  Avg not_seen: {np.mean(not_seen_counts[-20:]):.1f}")
    print(f"  Avg eval_targets: {np.mean(eval_targets[-20:]):.1f}")

In [ ]:
# Plot preference extraction over time
if conversations:
    fig2 = make_subplots(rows=1, cols=2, subplot_titles=('Preferences Extracted', 'Eval Targets'))
    
    episodes = list(range(1, len(liked_counts) + 1))
    
    fig2.add_trace(
        go.Scatter(x=episodes, y=liked_counts, mode='lines', name='Liked', line=dict(color='green')),
        row=1, col=1
    )
    fig2.add_trace(
        go.Scatter(x=episodes, y=disliked_counts, mode='lines', name='Disliked', line=dict(color='red')),
        row=1, col=1
    )
    fig2.add_trace(
        go.Scatter(x=episodes, y=not_seen_counts, mode='lines', name='Not Seen', line=dict(color='gray')),
        row=1, col=1
    )
    
    fig2.add_trace(
        go.Scatter(x=episodes, y=eval_targets, mode='lines', name='Eval Targets', line=dict(color='purple')),
        row=1, col=2
    )
    
    fig2.update_layout(height=400, title_text='Preference Extraction Analysis')
    fig2.update_xaxes(title_text='Episode')
    fig2.show()

In [ ]:
# Show recent conversations
print("Recent conversations (last 5):")
print("=" * 80)
for conv in conversations[-5:]:
    print(f"\nEpisode {conv['episode']} | User {conv['user_id']} | Targets: {conv['num_eval_targets']}")
    print(f"  Reward: {conv['reward']:.4f} | NDCG: {conv['ndcg']:.4f}")
    prefs = conv.get('preferences', {})
    print(f"  Liked: {prefs.get('liked', [])}")
    print(f"  Disliked: {prefs.get('disliked', [])}")
    print(f"  Not seen: {prefs.get('not_seen', [])}")

In [ ]:
# Summary statistics
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)

# Compare first 10 vs last 10 episodes
if len(df) >= 20:
    early = df.head(10)
    late = df.tail(10)
    
    print(f"\n{'Metric':<15} {'Early (1-10)':<15} {'Recent (last 10)':<15} {'Delta':<10}")
    print("-" * 55)
    
    for col in ['reward', 'ndcg', 'loss']:
        early_val = early[col].mean()
        late_val = late[col].mean()
        delta = late_val - early_val
        print(f"{col:<15} {early_val:<15.4f} {late_val:<15.4f} {delta:+.4f}")
else:
    print(f"\nNeed at least 20 episodes for comparison (have {len(df)})")

print(f"\nTotal episodes: {len(df)}")